# Large-Scale Weather and Climate Data Processing — results

Reads the small CSV summaries produced by the Spark pipeline (`data/export/`) and draws the
figures for the report. Heavy processing happened on the cluster; this notebook only plots.

Run the pipeline first: `bash scripts/run_pipeline.sh` (cluster) or
`bash scripts/run_pipeline.sh --local --sample` (laptop test).

In [ ]:
import glob, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPORT = REPO / "data" / "export"           # use REPO / "data/sample/export" for the sample run
if not EXPORT.exists():
    EXPORT = REPO / "data" / "sample" / "export"
print("Reading from", EXPORT)

def load(name):
    files = glob.glob(str(EXPORT / name / "*.csv"))
    return pd.read_csv(files[0]) if files else pd.DataFrame()

FOCUS = json.load(open(REPO / "config" / "pipeline.json"))["focus_countries"]
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.grid": True, "grid.alpha": 0.3})

## 1. Data quality — what the cleaning removed

In [ ]:
status = load("quality_record_status")
pivot = status.pivot_table(index="element", columns="status", values="count", aggfunc="sum").fillna(0)
share = pivot.div(pivot.sum(axis=1), axis=0) * 100
display(pivot.astype(int))
display(share.round(3))
display(load("quality_coverage_by_year"))
display(load("quality_qflag_breakdown").head(15))

## 2. Regional temperature anomalies (area-weighted)

In [ ]:
reg = load("regional_annual")
fig, ax = plt.subplots()
for r in ["Global", "Northern Hemisphere", "Southern Hemisphere"]:
    d = reg[reg.region == r].sort_values("year")
    ax.plot(d.year, d.tmean_anom_c, marker="o", label=r)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Anomaly vs baseline (°C)"); ax.set_title("Annual mean temperature anomaly"); ax.legend()

g = reg[reg.region == "Global"].dropna(subset=["tmean_anom_c"])
slope, intercept = np.polyfit(g.year, g.tmean_anom_c, 1)
print(f"Global OLS trend over {g.year.min()}-{g.year.max()}: {slope*10:+.2f} °C/decade "
      f"(descriptive only — a short window is dominated by year-to-year variability such as ENSO)")

## 3. Focus countries — annual anomalies and hot days

In [ ]:
ca = load("country_annual")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for c in FOCUS:
    d = ca[ca.country_code == c].sort_values("year")
    if d.empty: continue
    name = d.country_name.iloc[0]
    axes[0].plot(d.year, d.tmean_anom_c, marker="o", label=name)
    axes[1].plot(d.year, d.avg_hot_days, marker="o", label=name)
axes[0].set_title("Temperature anomaly (°C)"); axes[1].set_title("Avg days with TMAX ≥ 35 °C per station")
axes[0].legend(); plt.tight_layout()

## 4. Seasonal cycle (baseline climatology) and seasonal anomalies

In [ ]:
clim = load("country_monthly_climatology")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for c in FOCUS:
    d = clim[clim.country_code == c]
    if d.empty: continue
    ax[0].plot(d.month, d.clim_tmean_c, marker="o", label=d.country_name.iloc[0])
    ax[1].bar(d.month + 0.12 * FOCUS.index(c), d.clim_prcp_mm, width=0.12, label=d.country_name.iloc[0])
ax[0].set_title("Monthly mean temperature (°C)"); ax[1].set_title("Monthly precipitation (mm)")
ax[0].legend(); plt.tight_layout()

cs = load("country_seasonal")
focus_seasonal = cs[cs.country_code.isin(FOCUS)].pivot_table(
    index=["country_name", "season"], columns="season_year", values="tmean_anom_c")
display(focus_seasonal.round(2))

## 5. Spatial pattern — station warming trends

In [ ]:
tr = load("station_trends").dropna(subset=["tmean_anom_c_trend_per_decade"])
lim = tr.tmean_anom_c_trend_per_decade.quantile([0.02, 0.98]).abs().max()
fig, ax = plt.subplots(figsize=(13, 6))
sc = ax.scatter(tr.longitude, tr.latitude, c=tr.tmean_anom_c_trend_per_decade, cmap="RdBu_r",
                vmin=-lim, vmax=lim, s=6)
plt.colorbar(sc, label="°C per decade"); ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_title(f"Station temperature trends ({len(tr):,} stations)")
print(tr.tmean_anom_c_trend_per_decade.describe())

In [ ]:
grid = load("grid_annual")
last = grid[grid.year == grid.year.max()]
fig, ax = plt.subplots(figsize=(13, 6))
sc = ax.scatter(last.cell_lon, last.cell_lat, c=last.tmean_anom_c, cmap="RdBu_r", vmin=-3, vmax=3,
                marker="s", s=60)
plt.colorbar(sc, label="°C"); ax.set_title(f"Grid-cell anomaly, {int(grid.year.max())}")

## 6. Extremes

In [ ]:
display(load("top_hottest_days").head(15))
display(load("top_coldest_days").head(10))
display(load("top_wettest_days").head(10))
rec = load("country_records")
display(rec[rec.country_code.isin(FOCUS)])

## 7. Scaling: 1 worker vs 2 workers

In [ ]:
bench = REPO / "logs" / "benchmark.csv"
if bench.exists():
    b = pd.read_csv(bench)
    s = b.groupby("workers").seconds.agg(["mean", "std", "count"])
    display(s)
    s["mean"].plot.bar(yerr=s["std"], title="Runtime by number of workers", ylabel="seconds")
    print(f"Speed-up with 2 workers: {s.loc[1, 'mean'] / s.loc[2, 'mean']:.2f}x")
else:
    print("Run scripts/benchmark.sh on the master first.")

## 8. Descriptive vs causal — how to read these results

Everything above is **descriptive**: averages, anomalies, extremes and linear trends of what was
observed at stations. It tells us *what changed and where*, not *why*.

* A positive trend at a station is consistent with global warming but could also come from
  urban growth around the station, an instrument change, a station move or natural variability
  (ENSO, NAO) — GHCN-Daily is **not homogenised** for these.
* A ~10-year window is short; year-to-year variability is as large as the trend.
* Attribution (saying how much of a change is *caused* by greenhouse gases) needs climate models,
  counterfactual simulations and formal detection-attribution methods — outside this project.
* Station coverage is uneven (dense in the USA/Europe, sparse in Africa, including Libya), so
  global means are weighted per grid cell and cells with few stations are less reliable.